# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

The dataset may contain one or more record sets. Each record set can be accessed by its `@id`.

In [ ]:
from pprint import pprint

# List available RecordSets by their @id
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
print("Available RecordSet @ids:")
pprint(record_set_ids)

# Print fields for each RecordSet
for rs_json in dataset.metadata.to_json().get('recordSet', []):
    print(f"\nRecordSet @id: {rs_json.get('@id')}")
    if 'field' in rs_json:
        fields = rs_json['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            # field can be a @id or an expanded dict, handle both
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"  Field @id: {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we load **all available record sets** into individual DataFrames for further processing.

In [ ]:
# Select the available record sets
record_sets = record_set_ids  # From previous cell
dataframes = {}

for record_set_id in record_sets:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for RecordSet '{record_set_id}' with columns:")
    print(df.columns.tolist())
    if not df.empty:
        display(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
You may want to focus on a main tabular record set. Below we select the first available record set and numeric field for illustration.

In [ ]:
import numpy as np

# Pick the primary record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    raise ValueError('No RecordSets found in schema.')

df = dataframes[main_record_set_id]

print(f"Main RecordSet: {main_record_set_id}. Available columns:")
print(list(df.columns))

# Find a likely numeric field. GAD-7, PHQ-9, or PSQ total scores are likely numeric
numeric_candidates = [col for col in df.columns if ('gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower() or 'score' in col.lower() or df[col].dtype in ['int64','float64'])]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # fallback: first numeric column
    numeric_field = df.select_dtypes(include=np.number).columns[0] if not df.select_dtypes(include=np.number).empty else df.columns[0]

print(f"Using the field '{numeric_field}' as the numeric field for analysis.")

# Filter for values greater than an arbitrary threshold (e.g., above average)
if np.issubdtype(df[numeric_field].dtype, np.number):
    mean_value = df[numeric_field].mean()
    threshold = mean_value
    filtered_df = df[df[numeric_field] > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df[[numeric_field]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a common categorical field if it exists
    candidates = ['village', 'Village', 'gender', 'Gender', 'marital_status', 'age_group', 'Age']
    group_field = None
    for c in candidates:
        if c in df.columns:
            group_field = c
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean '{numeric_field}' grouped by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No group field found for grouping.")
else:
    print(f"Selected field '{numeric_field}' is not numeric. Please select another field.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

if np.issubdtype(df[numeric_field].dtype, np.number):
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group field (if available)
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"Cannot plot: '{numeric_field}' is not numeric.")

## 6. Conclusion
In this notebook, you explored demographic and mental health survey data from the Kilifi County FAIR² dataset using `mlcroissant`.

- We loaded and overviewed all available record sets and field `@id`s as per Croissant schema standards.
- We extracted tabular data and illustrated filtering and normalization of a numeric mental health score field.
- We used group-wise means and visualized the distribution and differences by group (e.g., by village or gender).

For deeper analysis, see the dataset's full documentation and RAI statement, and extend this workflow to combine multiple record sets or join with external data as allowed.